# Anthropic SDK 连通性测试

通过本地代理端点 `http://192.168.1.67:9443/anthropic` 测试 Anthropic SDK 是否可用。

**模型**: `deepseek-v4-pro`

## 1. 导入与配置

In [1]:
from anthropic import Anthropic, NotFoundError, AuthenticationError, APIStatusError

# 代理端点配置
BASE_URL = "http://192.168.1.67:9443/anthropic"
API_KEY = "c5930172761e6415e2d2e1ddc5f74108"

print(f"端点: {BASE_URL}")

端点: http://192.168.1.67:9443/anthropic


## 2. 初始化客户端

In [2]:
client = Anthropic(base_url=BASE_URL, api_key=API_KEY)
print("客户端初始化完成")

客户端初始化完成


## 3. 测试 /v1/models 端点

> 注意：很多代理不暴露此端点，404 属于正常现象。

In [5]:
try:
    models = client.models.list()
    model_ids = [m.id for m in models.data]
    print(f"可用模型: {model_ids}")
except NotFoundError:
    print("代理未暴露 /v1/models（正常现象）")
except Exception as e:
    print(f"异常: {type(e).__name__}: {e}")

代理未暴露 /v1/models（正常现象）


## 4. 测试 /v1/messages 端点

发送一条最简单的消息，验证整个链路是否通畅。

> 关键细节：响应中可能包含 `ThinkingBlock`（推理过程），需要按 `b.type == "text"` 过滤才能拿到文本回复。

In [6]:
MODEL = "deepseek-v4-pro"

message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "用一句话介绍你自己"}],
)

# 过滤出文本块（跳过 thinking block）
text_blocks = [b for b in message.content if b.type == "text"]
reply = text_blocks[0].text.strip() if text_blocks else "（无文本回复）"

# 统计 thinking block 数量
thinking_count = sum(1 for b in message.content if b.type == "thinking")

print(f"模型: {message.model}")
print(f"回复: {reply}")
if thinking_count:
    print(f"（响应包含 {thinking_count} 个 thinking 块）")
print(f"Token 用量: input={message.usage.input_tokens}, output={message.usage.output_tokens}")

模型: deepseek-v4-pro
回复: 我是由深度求索公司创造的AI助手，热情、细腻，免费为大家提供各类帮助，尤其擅长处理长文本、文件和复杂任务。
（响应包含 1 个 thinking 块）
Token 用量: input=8, output=122


## 5. 完整使用示例

以下是最简洁的调用模板，可直接复制使用。

In [ ]:
from anthropic import Anthropic

# 初始化客户端
client = Anthropic(
    base_url="http://192.168.1.67:9443/anthropic",
    api_key="c5930172761e6415e2d2e1ddc5f74108",
)

# 发送消息
message = client.messages.create(
    model="deepseek-v4-pro",
    max_tokens=4096,
    messages=[{"role": "user", "content": "你的问题"}],
)

# 获取文本回复（跳过 thinking block）
text = "".join(b.text for b in message.content if b.type == "text")
print(text)